# Project 1: Group 42

Text classification on **SST-2** (binary sentiment). Task: `project.ipynb`. Code reused from `tutorial.ipynb` and `part1/*.py`.

Plan (section N = backup slide N):
1. Data and metric
2. Tokenizer
3. Representations: indicator, bag-of-words, + our choice
4. Architectures: log-linear, MLP, + our choice
5. Evaluation: cross-validation setup and metric
6. Analysis: Bayesian comparison with a ROPE, error analysis

> Use of AI tools: notebook scaffold (structure, setup/experiment/analysis helper code, TODO prompts) generated with Claude; implementations, experiments and conclusions are ours.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
from pathlib import Path
from collections import Counter, OrderedDict

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

root = Path.cwd().parent
sys.path.insert(0, str(root))
import part1

results_folder = Path("./results/p1_group42")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data and metric

SST-2 ([course dataset list](https://nlp-illc.github.io/nlp1/2026/datasets/)): `stanfordnlp/sst2`.
The test split has no labels (all -1), so we cross-validate on train and keep validation as our held-out test set.

In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")
print(dataset)

outcome_space = part1.clslib.NamedClasses(['negative', 'positive'])
print(outcome_space)

train_pairs = list(zip(dataset["train"]["sentence"], dataset["train"]["label"]))
heldout_pairs = list(zip(dataset["validation"]["sentence"], dataset["validation"]["label"]))   # held-out, use only at the very end

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
0: negative, 1: positive


In [8]:
print("Train Size: ", len(train_pairs))
print("Validation Size: ", len(heldout_pairs),"\n")

def label_stats(pairs, name):
    print(f"{name} labels:")
    labels = [label for text, label in pairs]
    labels = np.array(labels)
    print(" Positive: ", np.sum(labels))
    print(" Negative: ", len(pairs)-np.sum(labels))
    print(" Positive fraction: ", np.mean(labels))
    print(" Negative fraction: ", 1-np.mean(labels))
    print(" Majority baseline (accuracy):", max(np.mean(labels), 1 - np.mean(labels)),"\n")

#Train labels:
label_stats(train_pairs, "Train")

#Validation labels:
label_stats(heldout_pairs, "Validation")

# SST-2 train consists of phrases of sentences (see 5a), the held-out set of full sentences
print("average words per text: train %.1f, held-out %.1f" % (np.mean([len(t.split()) for t, _ in train_pairs]),
                                                           np.mean([len(t.split()) for t, _ in heldout_pairs])))

Train Size:  67349
Validation Size:  872 

Train labels:
 Positive:  37569
 Negative:  29780
 Positive fraction:  0.5578256544269403
 Negative fraction:  0.4421743455730597
 Majority baseline (accuracy): 0.5578256544269403 

Validation labels:
 Positive:  444
 Negative:  428
 Positive fraction:  0.5091743119266054
 Negative fraction:  0.49082568807339455
 Majority baseline (accuracy): 0.5091743119266054 



Metric. Classes are mildly imbalanced (55.8% positive in train, 50.9% in held-out), so accuracy is meaningful and easy to interpret; we use it as the main metric and compare against the majority baseline (0.558 CV / 0.509 held-out). We also report macro-F1 on the held-out set as a check. If it stays close to accuracy, the imbalance does not distort our conclusions.

## 2. Tokenizer

We train our own BPE tokenizer on the SST-2 train split. The sanity checks below compare vocabulary sizes and a tokenizer trained on AG News (the tutorial's corpus); motivation after the checks.

In [ ]:
vocab_size = 4000
min_frequency = 2
tokenizer_corpus = [text for text, _ in train_pairs]   # train split only: the held-out set is never seen

tokenizer = part1.experiment.train_tokenizer(tokenizer_corpus, vocab_size, min_frequency)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}_mf{min_frequency}.json")

print(tokenizer.encode(train_pairs[0][0]).tokens)

In [ ]:
# sanity checks on the held-out sentences, which no tokenizer has seen
heldout_words = [w for text, _ in heldout_pairs for w in text.split()]
print("distinct words in train:", len(set(w for text, _ in train_pairs for w in text.split())))

def tokenizer_stats(tok, name):
    pieces = np.array([len(tok.encode(w).tokens) for w in heldout_words])
    print(f"{name}: vocab {tok.get_vocab_size()}, {pieces.mean():.2f} tokens per word, {np.mean(pieces == 1):.1%} of words kept whole")

for vs in [1000, 2000, 4000, 8000, 16000]:
    tokenizer_stats(part1.experiment.train_tokenizer(tokenizer_corpus, vs, min_frequency), f"SST-2 BPE, vocab_size={vs}")
tokenizer_stats(part1.experiment.train_tokenizer(part1.experiment.download_ag_news_texts(), vocab_size, min_frequency), f"AG News BPE, vocab_size={vocab_size}")

unk_id = tokenizer.token_to_id("[UNK]")
print("[UNK] tokens in held-out:", sum(tokenizer.encode(text).ids.count(unk_id) for text, _ in heldout_pairs))
for text, _ in heldout_pairs[:3]:
    print(tokenizer.encode(text).tokens)

**Tokenizer choices** (measured on the held-out sentences above):
- *Own BPE instead of the tutorial's AG News one.* With the same vocabulary size, the AG News tokenizer keeps only 55.7% of the held-out words whole (1.77 tokens per word), ours 79.4% (1.35). News and movie reviews use different words, and SST-2 is lowercased and pre-tokenized (`n't`, `'s`).
- *vocab_size = 4000.* Train has 14,816 distinct words. A larger vocabulary keeps more words whole (8000: 86.8%), but it doubles the number of features: the encoders return dense vectors, so a training fold is 54k x vocab_size floats (1.7 GB at 4000), and there are more rare features to overfit on. We did not tune it with CV (compute).
- *min_frequency = 2*: no merges for strings seen only once.
- *Corpus:* the train split only. It is small, but it is the right domain, and only 3 `[UNK]` tokens occur in the held-out set. Train contains overlapping phrases, so words are counted several times; this changes the merge counts, not which pieces exist.
- The tokenizer is trained on all of train before the CV split, so it sees the text (not the labels) of the validation folds. This is a mild optimistic bias that we accept; the held-out set is never seen.

## 5a. Cross-validation splits

(Section 5 of the plan; created here because sections 3 and 4 need the splits.)

The SST-2 train split is not a set of independent sentences: it contains every phrase of every sentence's parse tree
(e.g. "the film is n't good" but also "is n't good" and "good"). With a random split, pieces of the same sentence end up
in both the training and the validation fold, which makes CV accuracy too optimistic. So we split by sentence:
every phrase is assigned to the longest train text that contains it (its source sentence), and all phrases of a
sentence go to the same fold.

In [ ]:
import random

def sentence_groups(texts):
    """Map every phrase to the index of the longest train text (= source sentence) that contains it."""
    groups = [None] * len(texts)
    span_to_group = {}
    for i in sorted(range(len(texts)), key=lambda i: -len(texts[i].split())):
        words = texts[i].split()
        if ' '.join(words) in span_to_group:
            groups[i] = span_to_group[' '.join(words)]
        else:  # not part of a longer text: a new sentence, register all its word spans
            groups[i] = i
            for a in range(len(words)):
                for b in range(a + 1, len(words) + 1):
                    span_to_group.setdefault(' '.join(words[a:b]), i)
    return groups

def grouped_cross_validation_split(pairs, groups, k, seed=0):
    """Like part1.experiment.cross_validation_split, but all phrases of a sentence go to the same fold."""
    phrases_of = {}
    for i, g in enumerate(groups):
        phrases_of.setdefault(g, []).append(i)
    group_ids = list(phrases_of)
    random.seed(seed)
    random.shuffle(group_ids)
    folds = [[i for g in part for i in phrases_of[g]] for part in np.array_split(group_ids, k)]

    cross_validation_sets = {}
    for i in range(k):
        training_ids = [j for f in range(k) if f != i for j in folds[f]]
        cross_validation_sets[i] = {
            "training": [pairs[j] for j in training_ids],
            "validation": [pairs[j] for j in folds[i]],
        }
    return cross_validation_sets

groups = sentence_groups([text for text, _ in train_pairs])
print("phrases:", len(groups), "sentences:", len(set(groups)))

my_seeds = [0, 1, 2]   # 3 seeds x 5 folds = 15 measurements per system
num_folds = 5

my_cross_validation_sets = [
    grouped_cross_validation_split(pairs=train_pairs, groups=groups, k=num_folds, seed=seed)
    for seed in my_seeds
]

## 3. Representations

Note: `part1.clslib.TokenIndicator.__len__` uses the global variable `tokenizer` (a bug in the course code).
It works as long as our tokenizer is called `tokenizer`; the copy below fixes it.

In [ ]:
from tokenizers import Tokenizer

class TokenIndicator(part1.clslib.Encoder):
    """h_v(x) = 1 if BPE token v occurs in x"""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self):
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        vector[list(set(self.tokenizer.encode(x).ids))] = 1
        return vector

In [ ]:
class BagOfWords(part1.clslib.Encoder):
    """h_v(x) = number of times BPE token v occurs in x"""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self):
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector

In [ ]:
class TfIdf(part1.clslib.Encoder):
    """h_v(x) = (number of times BPE token v occurs in x) * idf(v)
    idf(v) = log((1 + N) / (1 + number of sentences in corpus that contain v))
    """

    def __init__(self, tokenizer: Tokenizer, corpus: list):
        self.tokenizer = tokenizer
        df = np.zeros(len(self))
        for text in corpus:
            df[list(set(tokenizer.encode(text).ids))] += 1
        self.idf = np.log((1 + len(corpus)) / (1 + df))

    def __len__(self):
        return self.tokenizer.get_vocab_size()

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector * self.idf

In [ ]:
encoders = {
    'indicator': TokenIndicator(tokenizer),
    'bow': BagOfWords(tokenizer),
    'tfidf': TfIdf(tokenizer, tokenizer_corpus),   # idf from the same corpus as the tokenizer (no labels used)
}
# sanity check: dimensions and an example
for name, enc in encoders.items():
    h = enc(train_pairs[0][0])
    print(name, len(enc), h.sum())

**Representations.** Indicator and bag-of-words differ only when a token occurs more than once in a text, and in short phrases (9.4 words on average) that is the case for only 29% of the train texts, so we expect them to perform about the same.
TF-IDF is our own choice: it gives less weight to tokens that occur in almost every text (`.`, `the`, `a`) and more to rare, more specific ones. We do not normalise the vectors, so the features are scaled up by the idf (up to about 11).
The idf is computed on the same corpus as the tokenizer (no labels).

## 4. Architectures and experiments

`part1.experiment.cross_validation_experiment` supports `model_option='logistic'` (log-linear), `'mlp'` (1 hidden layer)
and `'naive_bayes'` (our third architecture, see `nb.py`).

In [ ]:
my_classifiers = OrderedDict()
scores_folder = Path("./p1_scores")   # tracked in git: fold accuracies only, no models
scores_folder.mkdir(exist_ok=True)

def run(name, encoder, model_option, **kwargs):
    """Cross-validate one system for every seed; results go to my_classifiers[name] and p1_scores/{name}.json"""
    results = []
    for run_id, seed in enumerate(my_seeds):
        # seeds model training (saga in loglin, MLP init and batches), so results don't depend on which cells ran before
        np.random.seed(seed)
        torch.manual_seed(seed)
        results.append(
            part1.experiment.cross_validation_experiment(
                data_splits=my_cross_validation_sets[run_id],
                outcome_space=outcome_space,
                encoder=encoder,
                model_option=model_option,
                save_to=f"{results_folder}/{name}/seed-{seed}",
                **kwargs,
            )
        )
    my_classifiers[name] = results   # only stored if every run finished
    with open(scores_folder / f"{name}.json", "w") as f:
        json.dump([{'accuracies': {int(k): float(v) for k, v in r['accuracies'].items()}} for r in results], f)

def load_scores():
    """Load every teammate's pushed fold accuracies (systems not run in this kernel)."""
    for path in sorted(scores_folder.glob("*.json")):
        if path.stem not in my_classifiers:
            with open(path) as f:
                my_classifiers[path.stem] = json.load(f)

In [ ]:
%%time
for enc_name, enc in encoders.items():
    run(f"loglin_{enc_name}", enc, 'logistic', loglin_args={'max_iter': 100})

In [ ]:
%%time
# hidden_dim, dropout, batch_size, learning_rate, weight_decay: the tutorial's defaults (not tuned, see the notes below)
# num_epochs=5 with the best epoch picked on the dev split: the dev loss is lowest after epoch 1 in every run
mlp_args = {'hidden_dim': 128, 'dropout': 0.1, 'batch_size': 64, 'learning_rate': 1e-3,
            'weight_decay': 1e-4, 'num_epochs': 5,
            'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
for enc_name, enc in encoders.items():
    run(f"mlp_{enc_name}", enc, 'mlp', smlp_args=mlp_args)

In [ ]:
%%time
# third architecture: multinomial naive Bayes, alpha=1 is Laplace smoothing
for enc_name, enc in encoders.items():
    run(f"nb_{enc_name}", enc, 'naive_bayes', nb_args={'alpha': 1.0})

**Architectures.**
- *Log-linear* (tutorial): a discriminative model of p(y | x), linear in the features.
- *MLP with one hidden layer* (tutorial): can model interactions between features. The hyperparameters are the tutorial's defaults, we did not tune them. The dev loss is lowest after the first epoch in all 45 runs, so the MLP overfits fast (train accuracy goes from about 0.86 to 0.97 in 5 epochs); more dropout or a smaller learning rate might help. It trains on 90% of each training fold, because 10% is the dev split for picking the epoch.
- *Multinomial naive Bayes* (our choice): a generative model of p(x | y) p(y) that assumes tokens are independent given the label. It trains in seconds and is a classic sentiment baseline; the comparison with log-linear is generative vs discriminative with the same features (Ng & Jordan, 2001: NB reaches its best error with less data, but that error is usually higher). alpha = 1 is the standard Laplace smoothing, not tuned.

## 5b + 6. Results and analysis

Slide 5: is the evaluation adequate (metric, CV setup)? Slide 6: which system is best, and how sure are we?

In [ ]:
load_scores()
df = part1.analysis.make_data_frame(my_classifiers)
display(df.groupby('model').accuracy.agg(['mean', 'std']).sort_values('mean', ascending=False))
part1.analysis.plot_box_and_points(df)
# majority-class baseline on the same validation folds: always predict 'positive' (the majority class in train)
majority = [np.mean([label for _, label in split['validation']]) for cv in my_cross_validation_sets for split in cv.values()]
print(f"majority baseline: {np.mean(majority):.4f} (std {np.std(majority):.4f})")

In [ ]:
# the same systems in an earlier run with a random (phrase-level) split instead of the split by sentence
# (in that run the MLP also picked its best epoch on the validation fold itself, with 10 epochs)
random_split = {}
for path in sorted(Path("./p1_scores_random_split").glob("*.json")):
    with open(path) as f:
        random_split[path.stem] = np.mean([list(run['accuracies'].values()) for run in json.load(f)])
split_effect = pd.DataFrame({'random split': random_split, 'split by sentence': df.groupby('model').accuracy.mean()})
split_effect['difference'] = split_effect['random split'] - split_effect['split by sentence']
display(split_effect.round(4))

**Evaluation (slide 5).**
- All systems are far above the majority baseline (0.558 on the CV folds).
- *The split matters most.* With a random phrase-level split, the CV accuracy is 4-11 points higher, because phrases of the same sentence end up in train and validation. The MLP gains the most (+10 to +11 points; it memorises best), which made it look 5 points better than log-linear; with the split by sentence that advantage almost disappears. For log-linear and NB only the split changed between the two runs, so their difference (4-8 points) is the effect of the leak alone.
- The split by sentence is confirmed by the held-out set: CV 0.811 vs held-out 0.803 for the final system (section 7).
- Accuracy is a reasonable metric here: the classes are almost balanced, and macro-F1 on the held-out set (0.802) is almost equal to accuracy (0.803).
- 3 seeds x 5 folds = 15 measurements per system; the correlated t-test accounts for the overlap between training folds.

In [ ]:
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

ROPE = 0.01   # one accuracy point, motivation below

def compare(a, b, rope=None, plot=False):
    rope = ROPE if rope is None else rope
    post = part1.analysis.BenavoliCorrelatedT(
        get_all_measurements_for(my_classifiers, a).flatten(),
        get_all_measurements_for(my_classifiers, b).flatten(),
        runs=len(my_seeds),   # the tutorial omits this; with several seeds it must be set
        names=[a, b],
    )
    p_a, p_tie, p_b = post.probs(rope)
    if plot:
        post.plot(rope)
    return {'A': a, 'B': b, 'mean diff (B-A)': post.mean_diff, 'P(A wins)': p_a, 'P(tie)': p_tie, 'P(B wins)': p_b}

def all_comparisons(rope=None):
    rows = []
    for rep in ['indicator', 'bow', 'tfidf']:      # effect of the architecture, for a fixed representation
        for a, b in [('loglin', 'mlp'), ('loglin', 'nb'), ('mlp', 'nb')]:
            rows.append(compare(f"{a}_{rep}", f"{b}_{rep}", rope))
    for arch in ['loglin', 'mlp', 'nb']:           # effect of the representation, for a fixed architecture
        for a, b in [('indicator', 'bow'), ('indicator', 'tfidf'), ('bow', 'tfidf')]:
            rows.append(compare(f"{arch}_{a}", f"{arch}_{b}", rope))
    return pd.DataFrame(rows)

display(all_comparisons().round(3))

In [ ]:
# how much do the conclusions depend on the ROPE? the same comparisons with half the ROPE
display(all_comparisons(rope=0.005).round(3))

In [ ]:
compare('loglin_bow', 'mlp_bow', plot=True)
compare('loglin_tfidf', 'mlp_tfidf', plot=True);

**ROPE and results (slide 6).** We use a ROPE of 1 accuracy point:
- the fold-to-fold standard deviation of a single system is about 0.6 points, so smaller differences are of the order of the noise between data splits;
- on the 872 held-out sentences, 1 point is 9 sentences, and the 95% interval of a held-out accuracy around 0.80 is about ±2.7 points, so smaller differences could not be confirmed on the official test data anyway;
- in practice, a gain under 1 point does not justify a more expensive model (the MLP takes about 7 minutes per CV run on a GPU, log-linear and NB under a minute).

Findings with ROPE = 0.01:
- *Architecture:* MLP and log-linear are practically equivalent on indicator and BoW (P(tie) ≥ 0.998). NB is about 1 point worse than the other two, which is right at the edge of the ROPE (P(tie) 0.17-0.67). The one clear difference is with TF-IDF: MLP beats log-linear (+1.7 points, P = 0.995) and NB (+2.8, P = 1.0).
- *Representation:* indicator and BoW are equivalent for every architecture (P(tie) = 1.0). TF-IDF is equivalent for the MLP, but about 1 point worse for log-linear and NB, at the edge of the ROPE (P(tie) 0.77-0.90 for log-linear, 0.20-0.39 for NB). A likely reason is the unnormalised idf scaling: it changes how strongly the fixed L2 penalty (log-linear) and the fixed smoothing alpha = 1 (NB) act, and gives rare tokens large values; the MLP's first layer can rescale the features.
- With a ROPE of 0.005 the same picture holds, but the ~1-point gaps (NB vs the others, TF-IDF for log-linear and NB) become likely losses (P 0.81-1.0); MLP vs log-linear on indicator and BoW stays mostly a tie (P(tie) 0.92 and 0.79), and MLP + TF-IDF vs MLP + indicator moves towards an MLP + TF-IDF win (P = 0.75).
- *Chosen system:* MLP + TF-IDF. It has the highest mean (0.811), it is the only system that clearly beats others, and it is practically equivalent to the other MLPs. If cost mattered, log-linear + BoW would be equally good in practice for a fraction of the compute.

## 7. Final evaluation on the held-out set

The held-out set (SST-2 validation split, 872 full sentences) is used once, for the system chosen above.

In [ ]:
# Final evaluation (run only once!): the chosen system, MLP + TF-IDF, trained on all of train and scored on the held-out set.
# As in CV, the best epoch is picked on a dev split (10% of the train sentences).
from sklearn.metrics import accuracy_score, f1_score

final_split = grouped_cross_validation_split(train_pairs, groups, k=10, seed=0)[0]
np.random.seed(0)
torch.manual_seed(0)
part1.smlp.train_neural_network_model(
    final_split["training"], final_split["validation"], len(outcome_space), encoders['tfidf'],
    save_to=f"{results_folder}/final_mlp_tfidf", **mlp_args
)
final_model = part1.smlp.load(f"{results_folder}/final_mlp_tfidf", device=mlp_args['device'])

heldout_texts = [text for text, _ in heldout_pairs]
heldout_gold = np.array([label for _, label in heldout_pairs])
heldout_probs = part1.smlp.predict_probs(heldout_texts, final_model, encoders['tfidf'], batch_size=256)
heldout_pred = heldout_probs.argmax(axis=1)

print(f"held-out accuracy: {accuracy_score(heldout_gold, heldout_pred):.4f}")
print(f"held-out macro-F1: {f1_score(heldout_gold, heldout_pred, average='macro'):.4f}")
print(f"majority baseline (always 'positive', the majority class in train): {np.mean(heldout_gold == 1):.4f}")

# predictions for the error analysis
with open("p1_heldout_predictions.json", "w") as f:
    json.dump([{'text': t, 'gold': int(g), 'pred': int(p), 'p_positive': float(pr[1])}
               for t, g, p, pr in zip(heldout_texts, heldout_gold, heldout_pred, heldout_probs)], f, indent=1)


In [ ]:
# error analysis on the held-out predictions of the final system (written by the cell above)
with open("p1_heldout_predictions.json") as f:
    heldout = pd.DataFrame(json.load(f))
heldout['error'] = heldout.gold != heldout.pred
heldout['words'] = heldout.text.str.split().str.len()
heldout['negation'] = heldout.text.str.contains(r"\b(?:not|no|never|nothing|n't)\b")
heldout['contrast'] = heldout.text.str.contains(r"\b(?:but|though|although|yet|however)\b")

print(f"errors: {heldout.error.sum()} of {len(heldout)}")
display(pd.crosstab(heldout.gold, heldout.pred, rownames=['gold'], colnames=['pred']))
display(heldout.groupby('negation').error.agg(['mean', 'size']))
display(heldout.groupby('contrast').error.agg(['mean', 'size']))
display(heldout.groupby(pd.cut(heldout.words, [0, 10, 20, 30, 100])).error.agg(['mean', 'size']))

In [ ]:
# the most confident mistakes
heldout['confidence'] = (heldout.p_positive - 0.5).abs()
for _, row in heldout[heldout.error].sort_values('confidence', ascending=False).head(10).iterrows():
    print(f"gold={row.gold} p_positive={row.p_positive:.3f} | {row.text}")

**Held-out result and error analysis.**
- MLP + TF-IDF: held-out accuracy 0.803, macro-F1 0.802 (majority baseline 0.509), close to its CV estimate of 0.811.
- 172 errors out of 872, slightly more false positives (96) than false negatives (76); the train set is 56% positive.
- *Negation:* 30.6% errors in sentences with not / no / never / nothing / n't vs 16.9% without.
- *Contrast:* 28.5% errors with but / though / yet / however vs 18.2% without.
- *Length:* the error rate grows from 15% (≤ 10 words) to 27% (> 30 words). Long sentences also contain more negations and contrasts, so these effects overlap.
- The most confident mistakes are exactly these cases ("the humor did n't quite engage this adult", "does n't waste a moment", "..., but he never really embraces ..."): a bag of tokens cannot tell what a negation or a "but" applies to, because word order is lost. Some labels are questionable ("routine, harmless diversion and little else" is labelled positive).
- This points to models that use word order and composition, the topic of the next projects.